In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import os
import zarr
os.chdir('/zhome/71/c/146676/main/')
from helpers import module_auxiliary as ma
from loaders import stitcher_XA
from loaders import loader_XA_to_NA
import SimpleITK as sitk

## Load 6.2 micron pixelsize recons and register to diffraction data

In [ ]:
with h5py.File("/dtu-compute/msaca/sliceA_diffraction/abs_volume/overview_abs_volume.h5", "r") as f:
    # List all groups and datasets
    print("Keys:", list(f.keys()))
    
    # Access a dataset (replace 'dataset_name' with an actual name from the keys)
    dataset = f["absorption"]
    DA = dataset[:,120:240]


In [ ]:
data = np.load('/dtu-compute/msaca/sliceA_xray_pc/resampled_to_diffraction/sampleA_xray.npy')
plt.imshow(DA[:,47])
plt.show()
plt.imshow(data[:,250])
plt.show()

In [ ]:
data = np.load('/dtu-compute/msaca/sliceA_xray_pc/resampled_to_diffraction/sampleA_neutron.npy')
plt.imshow(DA[:,50])
plt.show()
plt.imshow(data[:,250])
plt.show()

In [ ]:

# Open in read-only mode
root = zarr.open_group("/dtu-compute/msaca/sliceA_xray_pc/sliceA_reconstructions.ome.zarr", mode="r")

# Access the full-resolution arrays (level 0)
xray_lvl1 = root["xray/1"]
neutron_lvl0 = root["neutron/0"]

print("X-ray shape:", xray_lvl1.shape)
print("Neutron shape:", neutron_lvl0.shape)


In [ ]:

# Example: load a subvolume
# Let's say you want z=100:200, y=500:800, x=400:700
sub_xray = xray_lvl1[1000:1400, :, :]  # note: different resolution
sub_neutron = neutron_lvl0[1000:1400, :, :]  # note: different resolution

print("Sub X-ray shape:", sub_xray.shape)
print("Sub Neutron shape:", sub_neutron.shape)

In [ ]:
plt.imshow(DA[:,60])
plt.show()

plt.imshow(sub_xray[:,150])

In [ ]:
from scipy.ndimage import zoom
DA_u = zoom(DA, zoom=5, order=1)  # order=1 → trilinear interpolation


In [ ]:
class Registrator():

    def _downsample_image(self, input_image, factor=10,smooth=False):
        """
        Downsamples a 3D volume by reducing its resolution by a specified factor in each dimension,
        effectively averaging values over blocks (e.g., 2x2x2 cubes when factor=2).

        Args:
            input_image (sitk.Image): The input SimpleITK image to be downsampled.
            factor (int, optional): The downsampling factor. Defaults to 2.
                - A factor of 2 will halve the size of the image in each dimension.
                - The new spacing will be adjusted accordingly.

        Returns:
            sitk.Image: The downsampled SimpleITK image with reduced resolution.

        Function Steps:
            1. Retrieve the original size and spacing of the input image.
            2. Compute the new size by dividing each dimension of the image size by the factor.
            3. Compute the new spacing by multiplying the original spacing by the factor.
            4. Use `sitk.Resample` to resample the image:
                - Use an identity transform to retain the original alignment.
                - Apply `sitk.sitkBSpline` interpolation to achieve smooth averaging.
            5. Return the downsampled image.

        Example:
            # Load a 3D image
            input_image = sitk.ReadImage('large_image.nii')
            
            # Downsample the image to half the size in each dimension
            downsampled_image = downsample_image(input_image, factor=2)
            
            # Save the downsampled image
            sitk.WriteImage(downsampled_image, 'downsampled_image.nii')
        """
        if smooth:
            mean_filter = sitk.MeanImageFilter()
            mean_filter.SetRadius(smooth)  # Adjust the radius as needed
            image = mean_filter.Execute(input_image)
        else:
            image = input_image


        # Get the original size and spacing of the image
        size = image.GetSize()
        spacing = image.GetSpacing()

        # Calculate the new size (downsampling by factor)
        new_size = [int(size[0] / factor), int(size[1] / factor), int(size[2] / factor)]
        
        # Calculate the new spacing (enlarging the spacing to match the downsampled size)
        new_spacing = [s * factor for s in spacing]
        
        # Set the resampling transform (identity transform)
        transform = sitk.Transform(3, sitk.sitkIdentity)

        # Perform the resampling (using average interpolation for downsampling)
        downsampled_image = sitk.Resample(image,
                                        new_size,
                                        transform,
                                        sitk.sitkLinear,  # BSpline interpolation is good for downsampling
                                        image.GetOrigin(),
                                        new_spacing,
                                        image.GetDirection(),
                                        0)  # 0 is the background value for the resampling
        
        return downsampled_image


    def choose_registration_images(self, smoothing_i, shrinking_i, smooth_fixed, smooth_moving):
        if (shrinking_i is not None) and smooth_fixed:
            if str(smoothing_i) in self.fixed_smooth:
                print('Using precalculated smoothed fixed image with shrinking')
                fixed_d = self._downsample_image(self.fixed_smooth[str(smoothing_i)], factor=shrinking_i,smooth = 0)
            else:
                print('Calculating smoothed fixed image with shrinking')
                fixed_d = self._downsample_image(self.fixed, factor=shrinking_i,smooth=smoothing_i)

        if (shrinking_i is not None) and smooth_moving:
            if str(smoothing_i) in self.moving_smooth:
                print('Using precalculated smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving_smooth[str(smoothing_i)], factor=shrinking_i,smooth = 0)
            else:
                print('Calculating smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving, factor=shrinking_i,smooth=smoothing_i)


        if (shrinking_i is not None) and (not smooth_fixed):
            print('Using non-smoothed fixed image with shrinking')
            fixed_d = self._downsample_image(self.fixed, factor=shrinking_i,smooth=0)

        if (shrinking_i is not None) and (not smooth_moving):
                print('Using non-smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving, factor=shrinking_i,smooth=0)

        if (shrinking_i is None):
            print('Using non-smoothed moving image without shrinking')
            fixed_d = self.fixed
            moving_d = self.moving

        return fixed_d, moving_d

    def register(self, learning_rate = 0.1, sampling_percentage = 0.1,
                    max_iter = 50,
                    metric_type = 'ms', optimizer_type = 'gd',
                    smoothing = 0,
                    shrinking = 1, thresholds=None, histogram_bins = 50,
                    smooth_fixed = False, smooth_moving = False,mask = None,
                    mask_box = None, unit = 'index'):
        """
        Perform the image registration using the provided transformation parameters.
        
        Args:
            transform_params: The transformation parameters (e.g., translation, rotation).
            metric_type (str): The metric used for the registration (e.g., 'MeanSquares', 'Mattes').
            optimizer_type (str): The optimizer used for the registration (e.g., 'GradientDescent').
        
        Returns:
            SimpleITK.Transform: The resulting transformation after registration.
        """


        initial_transform = sitk.Similarity3DTransform()

        # Explicitly set the matrix to the identity matrix
        initial_transform.SetMatrix([1.0, 0.0, 0.0,
                                    0.0, 1.0, 0.0,
                                    0.0, 0.0, 1.0])

        self.metric_values = []
        self.iterations = []
        # Set the translation to zero
        initial_transform.SetTranslation([0.0, 0.0, 0.0])

        # Set the scale to 1.0
        initial_transform.SetScale(1.0)

        registration = sitk.ImageRegistrationMethod()

        self.first_metric_value = None
        self.second_metric_value = None

        registration.SetInitialTransform(initial_transform)

        fixed_d, moving_d = self.choose_registration_images(smoothing, shrinking, smooth_fixed, smooth_moving)


        if metric_type == 'mmi':
            registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=histogram_bins)
        elif metric_type == 'ms' or metric_type == 'ls':
            registration.SetMetricAsMeanSquares()
            fixed_d = sitk.BinaryThreshold(fixed_d, lowerThreshold=thresholds[0], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
            moving_d = sitk.BinaryThreshold(moving_d, lowerThreshold=thresholds[1], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
            fixed_d =sitk.Cast(fixed_d, sitk.sitkFloat32)
            moving_d =sitk.Cast(moving_d, sitk.sitkFloat32)

        else:
            raise ValueError('Specify metric type: Either "mmi" or "ms"/"ls"')

        if mask == 'fixed':
            mask = self.get_registration_mask(mask_box, which_image = 'fixed', unit = unit)
            registration.SetMetricFixedMask(mask)
        elif mask == 'moving':
            mask = self.get_registration_mask(mask_box, which_image = 'moving', unit = unit)
            registration.SetMetricFixedMask(mask)

        registration.SetMetricSamplingStrategy(registration.RANDOM)
        registration.SetMetricSamplingPercentage(sampling_percentage)

        if optimizer_type == 'gd':
            registration.SetOptimizerAsGradientDescent(
            learningRate=learning_rate,
            numberOfIterations=max_iter,
            convergenceMinimumValue=-1e-16,
            convergenceWindowSize=1000
        )


        else:
            raise ValueError('Specify optimization algorithm: Only "gd" currently supported')
                            
        #registration.SetOptimizerScalesFromPhysicalShift(smallParameterVariation = 0.01)
        registration.SetInterpolator(sitk.sitkLinear)


        self.metric_values = []
        self.iterations = []
            
        registration.Execute(fixed_d, moving_d)
        transform = registration.GetInitialTransform()
        initial_transform = sitk.Similarity3DTransform(transform)

        return transform

In [ ]:
fixed = sitk.GetImageFromArray(DA_u)
moving_small = sitk.GetImageFromArray(sub_xray)
moving_big = sitk.GetImageFromArray(sub_xray)


f_size = fixed.GetSize()
m_size = moving_small.GetSize()
max_f = max(f_size)
max_m = max(m_size)
max_size = max(max_f,max_m)



fixed.SetSpacing((1/max_f,1/max_f,1/max_f))
moving_small.SetSpacing((1.15/max_m,1.15/max_m,1.15/max_m))
moving_big.SetSpacing((1.15/max_m,1.15/max_m,1.15/max_m))


size_fixed = fixed.GetSize()
size_moving = moving_small.GetSize()
size_moving_big = moving_big.GetSize()

spacing_fixed = fixed.GetSpacing()
spacing_moving = moving_small.GetSpacing()
spacing_moving_big = moving_big.GetSpacing()

# Compute the new origin (shift it to -N/2)

# spatial size is roughly 1.5 cubed
# spacing is roughly 1.5/N_pixels
# Origin 
new_origin_fixed = [-0.5 * (size_fixed[i] - 1) * spacing_fixed[i] for i in range(len(size_fixed))]
new_origin_moving = [-0.5 * (size_moving[i] - 1) * spacing_moving[i] for i in range(len(size_moving))]
new_origin_moving_big = [-0.5 * (size_moving_big[i] - 1) * spacing_moving_big[i] for i in range(len(size_moving_big))]

# Set the origin to be the new center (-N/2)
fixed.SetOrigin(new_origin_fixed)
moving_small.SetOrigin(new_origin_moving)

moving_big.SetOrigin(new_origin_moving_big)


In [ ]:
reg = Registrator()
reg.fixed = fixed
reg.moving = moving_big
threshold = [0.0045,1.8]
# transform_registration= reg.register(learning_rate = 1, sampling_percentage = 1.0, max_iter = 100,
#                                         metric_type = 'ms', optimizer_type = 'gd',
#                                         smoothing = 0, shrinking = 5, thresholds=threshold,
#                                         smooth_fixed = False, smooth_moving = False)

resampled_moving_big = sitk.Resample(
    moving_big,                 # input image (fine resolution)
    fixed,                  # reference image -> defines resolution/grid
    transform_registration,              # your similarity3D transform
    sitk.sitkLinear,        # interpolator
    0.0,                    # default pixel value for out-of-bounds
    moving_small.GetPixelID()     # keep same pixel type as moving
)

In [ ]:
#np.save('/dtu-compute/msaca/sliceA_xray_pc/resampled_to_diffraction/sampleA_xray.npy', sitk.GetArrayFromImage(resampled_moving_big).astype(np.float32))

In [ ]:
px = (sitk.GetArrayFromImage(resampled_moving_big)[:,250,:]>1.8)*1.0
px2 = (sitk.GetArrayFromImage(resampled_moving_big)[:,250,:])*1.0
pn = (sitk.GetArrayFromImage(fixed)[:,250,:]>0.0045)*1.0
pn2 = (sitk.GetArrayFromImage(fixed)[:,250,:])*1.0
plt.imshow(pn,aspect='auto')
plt.show()
plt.imshow(pn2,aspect='auto')
plt.show()
plt.imshow(px,aspect='auto')
plt.show()
plt.imshow(px2,aspect='auto')
plt.show()

plt.imshow(pn-px,aspect='auto')

In [ ]:
# plot a vertical slice of DA

# load a few slices of the xray file, and find one that fits.
# Consider whether a simple flip transformation is needed
# take the absorption data and upscale it x5

# copy the registration code that was used to register the xray to the neuton data

# run te reistration loop and save the transformation

# apply the transformatoin to the 6.2 micron pixel data, both xray and neutron

In [ ]:
plt.imshow(sitk.GetArrayFromImage(resampled_moving_big)[:,300])
plt.show()

plt.imshow(sitk.GetArrayFromImage(resampled_moving_big)[300])
plt.show()



plt.imshow(sitk.GetArrayFromImage(fixed)[:,300])
plt.show()

plt.imshow(sitk.GetArrayFromImage(fixed)[300])
plt.show()

plt.imshow(DA[60])
plt.show()
